In [ ]:
from pathlib import Path
import platform

print("=== GOLDEN HOUR: GRANITE ENVIRONMENT ===")

# Verify system architecture
print("Architecture:", platform.machine())

# Check Granite model location
model_path = Path("../granite-4.0-h-350m-base-be.f16.gguf")

print("Model exists:", model_path.is_file())

if model_path.is_file():
    size_mb = model_path.stat().st_size / (1024 ** 2)
    print(f"Model size: {size_mb:.1f} MB")

# Check Python binding
try:
    import llama_cpp
    print("llama_cpp: AVAILABLE")
    print("Version:", getattr(llama_cpp, "__version__", "unknown"))
except Exception as e:
    print("llama_cpp: NOT AVAILABLE")
    print("Error:", str(e))


In [ ]:
import time
from llama_cpp import Llama

print("Loading IBM Granite...")
start = time.perf_counter()

llm = Llama(
    model_path=str(model_path),
    n_ctx=1024,
    n_threads=2,
    n_gpu_layers=0,
    verbose=False
)

load_time = time.perf_counter() - start

print("Granite loaded successfully!")
print(f"Loading time: {load_time:.2f} seconds")


In [ ]:

import time

prompt = """Convert account risk evidence into a short, factual explanation.
Do not invent facts or claim that money laundering is proven.

Account: A_EXAMPLE
Risk score: 0.85
Evidence: received 10 transfers from 4 flagged accounts in 1 hour
Explanation: Account A_EXAMPLE is recommended for analyst review because it received 10 transfers from 4 flagged accounts within one hour. This activity warrants investigation but does not prove money laundering.

Account: A_TEST
Risk score: 0.91
Evidence: received 14 transfers from 6 flagged accounts in 2 hours
Explanation:"""

start = time.perf_counter()

response = llm(
    prompt,
    max_tokens=100,
    temperature=0.1,
    top_p=0.9,
    stop=["\n\n", "\nAccount:"],
    echo=False
)

generation_time = time.perf_counter() - start

explanation = response["choices"][0]["text"].strip()

print("=== GRANITE EXPLANATION ===")
print(explanation)
print()
print(f"Generation time: {generation_time:.2f} seconds")
print("Generated tokens:", response["usage"]["completion_tokens"])


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("freeze_plan.csv")

print("File exists:", csv_path.exists())

if csv_path.exists():
    df = pd.read_csv(csv_path)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nNumber of accounts:", len(df))

    print("\nFirst 5 rows:")
    print(df.head().to_string(index=False))

    print("\nMissing values:")
    print(df.isnull().sum())


In [ ]:
import time

row = df.iloc[0]

account = str(row["acct"])
risk = float(row["risk_score"])
facts = str(row["reason_facts"])

print("Testing account:", account)

prompt = f"""Write a short, factual explanation for an analyst.
Use only the supplied evidence. Do not invent facts.
Do not claim that money laundering is proven.

Account: A_EXAMPLE
Risk score: 0.85
Evidence: received 10 transfers from 4 flagged accounts in 1 hour
Explanation: Account A_EXAMPLE is recommended for analyst review because it received 10 transfers from 4 flagged accounts within one hour. This pattern warrants investigation but does not prove money laundering.

Account: {account}
Risk score: {risk:.3f}
Evidence: {facts}
Explanation:"""

start = time.perf_counter()

response = llm(
    prompt,
    max_tokens=110,
    temperature=0.1,
    top_p=0.9,
    stop=["\n\n", "\nAccount:", "\nEvidence:"],
    echo=False
)

elapsed = time.perf_counter() - start
generated = response["choices"][0]["text"].strip()

print("\n=== GRANITE EXPLANATION ===")
print(generated)
print(f"\nGeneration time: {elapsed:.2f} seconds")
print("Generated tokens:", response["usage"]["completion_tokens"])
print("Finish reason:", response["choices"][0]["finish_reason"])


In [ ]:

import time

row = df.iloc[0]

account = str(row["acct"])
risk = float(row["risk_score"])
facts = str(row["reason_facts"])

prompt = f"""Task: Explain an account recommendation using only its recorded facts.
Do not invent time windows, transaction counts, amounts, or outcomes.
Do not describe money laundering as proven.

Example:
Account: A_DEMO
Evidence: received 8 transfers from 5 accounts; 3 connected accounts are flagged high-risk; activity spans 48 hours
Explanation: Account A_DEMO is recommended for analyst review because it received 8 transfers from 5 accounts and has 3 high-risk connected accounts. The recorded activity spans 48 hours. This warrants investigation but does not prove money laundering.

Now explain:
Account: {account}
Risk score: {risk:.3f}
Evidence: {facts}
Explanation:"""

start = time.perf_counter()

response = llm(
    prompt,
    max_tokens=130,
    temperature=0.0,
    stop=["\n\n", "\nAccount:", "\nEvidence:"],
    echo=False
)

elapsed = time.perf_counter() - start
generated = response["choices"][0]["text"].strip()

print("=== REVISED GRANITE EXPLANATION ===")
print(generated)
print(f"\nGeneration time: {elapsed:.2f} seconds")
print("Finish reason:", response["choices"][0]["finish_reason"])


In [ ]:

import re
import time

def generate_explanation(row):
    account = str(row["acct"])
    facts = str(row["reason_facts"])

    prompt = f"""Task: Explain an account recommendation using only its recorded facts.
Do not invent numbers, time windows, or outcomes.
Do not claim money laundering is proven.

Example:
Account: A_DEMO
Evidence: received 8 transfers from 5 accounts; 3 connected accounts are flagged high-risk; activity spans 48 hours
Explanation: Account A_DEMO is recommended for analyst review because it received 8 transfers from 5 accounts and has 3 high-risk connected accounts. Its activity spans 48 hours. This warrants investigation but does not prove money laundering.

Account: {account}
Evidence: {facts}
Explanation:"""

    start = time.perf_counter()

    response = llm(
        prompt,
        max_tokens=130,
        temperature=0.0,
        stop=["\n\n", "\nAccount:", "\nEvidence:"],
        echo=False
    )

    elapsed = time.perf_counter() - start
    explanation = response["choices"][0]["text"].strip()

    return explanation, elapsed, response["choices"][0]["finish_reason"]


def basic_validation(row, explanation, finish_reason):
    """Conservative checks; passing does not prove factual accuracy."""
    account = str(row["acct"])
    facts = str(row["reason_facts"])

    if not explanation:
        return False, "Empty explanation"

    if finish_reason == "length":
        return False, "Generation truncated"

    if account not in explanation:
        return False, "Missing account ID"

    # Reject account IDs other than the current account.
    ids = re.findall(r"\bA_[A-Za-z0-9]+\b", explanation)
    if any(x != account for x in ids):
        return False, "Unexpected account ID"

    # Detect numerical quantities absent from the source facts.
    # This is a heuristic, not a semantic fact checker.
    allowed = set(re.findall(r"\d+(?:\.\d+)?", facts))
    allowed.update(re.findall(r"\d+(?:\.\d+)?", str(row["risk_score"])))

    output_without_id = explanation.replace(account, "")
    numbers = re.findall(r"\d+(?:\.\d+)?", output_without_id)

    if any(n not in allowed for n in numbers):
        return False, "Potential unsupported number"

    if re.search(
        r"\b(confirmed laundering|proven guilty|definitely criminal)\b",
        explanation.lower()
    ):
        return False, "Unsupported certainty"

    return True, "Basic checks passed"


In [ ]:

from pathlib import Path

results = []

for _, row in df.iterrows():
    explanation, elapsed, finish_reason = generate_explanation(row)

    valid, check_result = basic_validation(
        row, explanation, finish_reason
    )

    results.append({
        "explanation": explanation,
        "generation_seconds": round(elapsed, 2),
        "validation_passed": valid,
        "validation_note": check_result
    })

    print(
        f"Account {row['acct']}: "
        f"{elapsed:.2f}s | {check_result}"
    )

# Preserve all original columns
df_explained = df.copy()
df_explained["explanation"] = [
    r["explanation"] for r in results
]

# Keep the review information in a separate CSV
review_df = df[["rank", "acct", "reason_facts"]].copy()
review_df["explanation"] = df_explained["explanation"]
review_df["generation_seconds"] = [
    r["generation_seconds"] for r in results
]
review_df["validation_passed"] = [
    r["validation_passed"] for r in results
]
review_df["validation_note"] = [
    r["validation_note"] for r in results
]

df_explained.to_csv(
    "freeze_plan_explained.csv",
    index=False
)

review_df.to_csv(
    "granite_explanation_review.csv",
    index=False
)

print("\n=== RESULTS ===")
print("Accounts processed:", len(results))
print("Basic checks passed:", sum(r["validation_passed"] for r in results))
print("Total generation time:",
      round(sum(r["generation_seconds"] for r in results), 2),
      "seconds")

print("Saved:", Path("freeze_plan_explained.csv").resolve())


In [ ]:

for _, row in review_df.iterrows():
    print("=" * 60)
    print("ACCOUNT:", row["acct"])
    print("ORIGINAL FACTS:", row["reason_facts"])
    print("GRANITE EXPLANATION:", row["explanation"])
    print("VALIDATION:", row["validation_note"])
    print()


In [ ]:

import pandas as pd

final_df = pd.read_csv("freeze_plan_explained.csv")

print("Rows:", len(final_df))
print("Columns:", final_df.columns.tolist())
print("Missing explanations:", final_df["explanation"].isna().sum())

assert len(final_df) == 10
assert "explanation" in final_df.columns
assert final_df["explanation"].notna().all()

print("\nFINAL CSV VERIFIED")


In [ ]:

import pandas as pd

demo_df = pd.read_csv("freeze_plan_explained.csv")

def show_account(rank=1):
    row = demo_df.loc[demo_df["rank"] == rank].iloc[0]

    print("GOLDEN HOUR — INVESTIGATOR REVIEW")
    print("=" * 45)
    print("Account:", row["acct"])
    print("Risk score:", round(row["risk_score"], 3))
    print("Original evidence:", row["reason_facts"])
    print()
    print("IBM GRANITE EXPLANATION:")
    print(row["explanation"])

show_account(1)


In [ ]:

import textwrap

def show_account(rank=1):
    row = demo_df.loc[demo_df["rank"] == rank].iloc[0]

    print("=" * 65)
    print("GOLDEN HOUR | INVESTIGATOR REVIEW")
    print("=" * 65)
    print(f"Account: {row['acct']}")
    print(f"Risk score: {row['risk_score']:.3f}")
    
    print("\nTRANSACTION EVIDENCE")
    print("-" * 65)
    print(textwrap.fill(str(row["reason_facts"]), width=65))

    print("\nIBM GRANITE EXPLANATION")
    print("-" * 65)
    print(textwrap.fill(str(row["explanation"]), width=65))

    print("\nDecision status: AWAITING INVESTIGATOR REVIEW")

show_account(1)


In [ ]:
import pandas as pd

latest = pd.read_csv("freeze_plan.csv")
explained = pd.read_csv("freeze_plan_explained.csv")

original_cols = [
    "rank", "acct", "risk_score",
    "money_blocked", "est_money_blocked",
    "reason_facts"
]

same = (
    latest[original_cols].reset_index(drop=True)
    .equals(explained[original_cols].reset_index(drop=True))
)

print("Freeze plan unchanged:", same)
print("Accounts:", len(latest))

In [ ]:

# Compare the latest freeze plan with your explained version

old = explained.set_index("acct")
new = latest.set_index("acct")

print("=== ACCOUNT COMPARISON ===")

print("New accounts:", sorted(set(new.index) - set(old.index)))
print("Removed accounts:", sorted(set(old.index) - set(new.index)))

common = old.index.intersection(new.index)

for acct in common:
    changes = []

    for col in original_cols:
        if col == "acct":
            continue

        a = old.loc[acct, col]
        b = new.loc[acct, col]

        if pd.isna(a) and pd.isna(b):
            continue

        if a != b:
            changes.append(col)

    if changes:
        print(f"{acct}: changed {', '.join(changes)}")

print("\nComparison complete.")


In [ ]:

import time

# Use Abdel's updated freeze plan
df = pd.read_csv("freeze_plan.csv")

# Check required columns before running Granite
required = {"acct", "reason_facts", "risk_score"}
assert required.issubset(df.columns)
assert df["acct"].is_unique

print("New accounts:", len(df))
print("Granite model ready:", llm is not None)

# Save the previous version as a backup
from pathlib import Path
import shutil

output = Path("freeze_plan_explained.csv")
if output.exists():
    shutil.copy2(output, "freeze_plan_explained_backup.csv")
    print("Previous output backed up.")

results = []

for _, row in df.iterrows():
    explanation, elapsed, finish_reason = generate_explanation(row)

    valid, note = basic_validation(
        row, explanation, finish_reason
    )

    results.append({
        "explanation": explanation,
        "generation_seconds": round(elapsed, 2),
        "validation_passed": valid,
        "validation_note": note
    })

    print(f"{row['acct']}: {elapsed:.2f}s | {note}")

# Preserve all of Abdel's latest columns
df_explained = df.copy()
df_explained["explanation"] = [
    r["explanation"] for r in results
]

# Save separate validation report
review_df = df[["acct", "reason_facts"]].copy()
review_df["explanation"] = df_explained["explanation"]
review_df["validation_passed"] = [
    r["validation_passed"] for r in results
]
review_df["validation_note"] = [
    r["validation_note"] for r in results
]

df_explained.to_csv(
    "freeze_plan_explained.csv",
    index=False
)

review_df.to_csv(
    "granite_explanation_review.csv",
    index=False
)

print("\n=== FINAL RESULTS ===")
print("Accounts processed:", len(results))
print("Basic checks passed:", sum(r["validation_passed"] for r in results))
print("Total generation time:",
      round(sum(r["generation_seconds"] for r in results), 2),
      "seconds")


In [ ]:

for _, row in review_df.iterrows():
    print("=" * 60)
    print("ACCOUNT:", row["acct"])
    print("FACTS:", row["reason_facts"])
    print("GRANITE:", row["explanation"])
    print("VALIDATION:", row["validation_note"])
    print()


In [ ]:

import re
import pandas as pd

path = "freeze_plan_explained.csv"
final_df = pd.read_csv(path)

problem_accounts = [
    "A_1abc58e5f795",
    "A_b53afc8551b3",
    "A_dd1f23b26aa3",
    "A_5dda8629fa3b"
]

# Keep a backup before corrections
final_df.to_csv(
    "freeze_plan_explained_before_corrections.csv",
    index=False
)

for acct in problem_accounts:
    mask = final_df["acct"] == acct

    if not mask.any():
        raise ValueError(f"Account missing: {acct}")

    explanation = final_df.loc[mask, "explanation"].iloc[0]

    explanation = re.sub(
        r",? (?:indicating|which is) a high-risk activity",
        "",
        explanation,
        flags=re.IGNORECASE
    )

    final_df.loc[mask, "explanation"] = explanation

final_df.to_csv(path, index=False)

print("Corrected explanations:")
for acct in problem_accounts:
    row = final_df.loc[final_df["acct"] == acct].iloc[0]
    print("\n", acct)
    print(row["explanation"])


In [ ]:

from pathlib import Path
import pandas as pd

v2_path = Path("guardian_v2_outputs/freeze_plan_candidate.csv")

print("V2 file exists:", v2_path.exists())

if v2_path.exists():
    v2 = pd.read_csv(v2_path)

    print("\nRows:", len(v2))
    print("\nColumns:", v2.columns.tolist())
    print("\nMissing values:")
    print(v2.isna().sum())

    print("\nFirst 3 rows:")
    print(v2.head(3).to_string(index=False))


In [ ]:

import time

row = v2.iloc[0]

account = str(row["acct"])
facts = str(row["reason_facts"])

prompt = f"""Task: Explain a recommended account for a financial investigator.
Use only the supplied facts. Keep the explanation concise.

Rules:
- Risk scores are account-level scores, not probabilities of crime.
- Suspicion proxies are estimates, not confirmed criminal money.
- Collateral proxies are uncalibrated estimates.
- Historical exposure is not money recovered or prevented.
- Do not invent facts or claim laundering is proven.
- All recommendations require human approval.

Example:
Account: A_DEMO
Evidence: Risk score 0.850 (account-level); 12 historical transfers; 4 counterparties; 3 cross-bank transfers; HUMAN REVIEW ONLY.
Explanation: Account A_DEMO is recommended for analyst review based on its account-level risk score of 0.850 and 12 historical transfers involving 4 counterparties, including 3 cross-bank transfers. These observations do not prove money laundering. Any intervention requires human approval.

Account: {account}
Evidence: {facts}
Explanation:"""

start = time.perf_counter()

response = llm(
    prompt,
    max_tokens=130,
    temperature=0.0,
    stop=["\n\n", "\nAccount:", "\nEvidence:"],
    echo=False
)

elapsed = time.perf_counter() - start
generated = response["choices"][0]["text"].strip()

print("=== GRANITE V2 EXPLANATION ===")
print(generated)
print(f"\nGeneration time: {elapsed:.2f} seconds")
print("Finish reason:", response["choices"][0]["finish_reason"])


In [ ]:

import pandas as pd
import time
from pathlib import Path

v2 = pd.read_csv(
    "guardian_v2_outputs/freeze_plan_candidate.csv"
)

def explain_v2(row):
    account = str(row["acct"])
    facts = str(row["reason_facts"])

    prompt = f"""Task: Explain an account recommendation using only recorded facts.
Use a factual and neutral tone.

Rules:
- Account risk scores are not probabilities of crime.
- Financial proxies are estimates, not confirmed criminal funds.
- Historical exposure is not money recovered or prevented.
- Never invent numbers, facts, or outcomes.
- Do not claim money laundering is proven.
- Any intervention requires human approval.

Example:
Account: A_DEMO
Evidence: Risk score 0.850 (account-level); 12 historical transfers; 4 counterparties; 3 cross-bank transfers; HUMAN REVIEW ONLY.
Explanation: Account A_DEMO is recommended for analyst review based on its account-level risk score of 0.850 and 12 historical transfers involving 4 counterparties, including 3 cross-bank transfers. These observations do not prove money laundering. Any intervention requires human approval.

Account: {account}
Evidence: {facts}
Explanation:"""

    start = time.perf_counter()

    response = llm(
        prompt,
        max_tokens=130,
        temperature=0.0,
        stop=["\n\n", "\nAccount:", "\nEvidence:"],
        echo=False
    )

    elapsed = time.perf_counter() - start

    return (
        response["choices"][0]["text"].strip(),
        elapsed,
        response["choices"][0]["finish_reason"]
    )


results = []

for _, row in v2.iterrows():
    explanation, seconds, finish = explain_v2(row)

    results.append({
        "explanation": explanation,
        "generation_seconds": round(seconds, 2),
        "finish_reason": finish
    })

    print(f"{row['acct']}: {seconds:.2f}s | {finish}", flush=True)

# Preserve all 11 V2 source columns
v2_explained = v2.copy()
v2_explained["explanation"] = [
    result["explanation"] for result in results
]

output = Path(
    "guardian_v2_outputs/freeze_plan_explained.csv"
)

v2_explained.to_csv(output, index=False)

print("\n=== V2 RESULTS ===")
print("Accounts:", len(v2_explained))
print("Missing explanations:",
      v2_explained["explanation"].eq("").sum())
print("Total inference seconds:",
      round(sum(r["generation_seconds"] for r in results), 2))
print("Saved:", output)


In [ ]:

for _, row in v2_explained.iterrows():
    print("=" * 60)
    print("ACCOUNT:", row["acct"])
    print("FACTS:", row["reason_facts"])
    print("GRANITE:", row["explanation"])
    print()


In [ ]:

import pandas as pd
from pathlib import Path
import shutil

path = Path("guardian_v2_outputs/freeze_plan_explained.csv")
backup = path.with_name("freeze_plan_explained_pre_review.csv")

shutil.copy2(path, backup)

df_final = pd.read_csv(path)

fix_accounts = [
    "A_ba6319e308ac",
    "A_974595b9f61a",
    "A_24cef1c2ae55",
    "A_17545e27b3d0"
]

for acct in fix_accounts:
    mask = df_final["acct"] == acct
    assert mask.sum() == 1, f"Account missing or duplicated: {acct}"

    row = df_final.loc[mask].iloc[0]

    # Use verified structured figures from the V2 source.
    import re

    facts = str(row["reason_facts"])

    tx = re.search(r"(\d+) historical transfers", facts)
    counterparties = re.search(r"(\d+) counterparties", facts)
    cross_bank = re.search(r"(\d+) cross-bank transfers", facts)

    assert tx and counterparties and cross_bank

    explanation = (
        f"Account {acct} is recommended for analyst review "
        f"based on its account-level risk score of "
        f"{float(row['risk_score']):.3f} and "
        f"{tx.group(1)} historical transfers involving "
        f"{counterparties.group(1)} counterparties, including "
        f"{cross_bank.group(1)} cross-bank transfers. "
        f"These observations do not prove money laundering. "
        f"Any intervention requires human approval."
    )

    df_final.loc[mask, "explanation"] = explanation

df_final.to_csv(path, index=False)

print("Corrected accounts:", len(fix_accounts))
print("Saved:", path)

for acct in fix_accounts:
    print("\n", df_final.loc[
        df_final["acct"] == acct, "explanation"
    ].iloc[0])


In [ ]:

source = pd.read_csv(
    "guardian_v2_outputs/freeze_plan_candidate.csv"
)

final = pd.read_csv(
    "guardian_v2_outputs/freeze_plan_explained.csv"
)

assert len(final) == len(source) == 10
assert source.equals(final[source.columns])
assert final["acct"].is_unique
assert final["explanation"].notna().all()
assert final["explanation"].str.strip().ne("").all()
assert final["explanation"].str.contains(
    "human approval", case=False
).all()

print("GUARDIAN V2 GRANITE OUTPUT VERIFIED")
print("Accounts:", len(final))
print("Source columns preserved:", len(source.columns))
print("Explanations:", final["explanation"].notna().sum())


In [ ]:

import pandas as pd

source = pd.read_csv(
    "guardian_v2_outputs/freeze_plan_candidate.csv"
)

final = pd.read_csv(
    "guardian_v2_outputs/freeze_plan_explained.csv"
)

print("Source rows:", len(source))
print("Final rows:", len(final))

print("\n=== COLUMN DIFFERENCES ===")

for col in source.columns:
    a = source[col].reset_index(drop=True)
    b = final[col].reset_index(drop=True)

    if a.equals(b):
        continue

    print(f"\nColumn: {col}")

    different = (
        ~(a.eq(b) | (a.isna() & b.isna()))
    )

    for i in different[different].index[:10]:
        print(f"Row {i}:")
        print("  Source:", repr(a.iloc[i]))
        print("  Final: ", repr(b.iloc[i]))

print("\n=== DATA TYPES ===")
print("Source:")
print(source.dtypes)
print("\nFinal:")
print(final[source.columns].dtypes)


In [ ]:

import pandas as pd
from pandas.testing import assert_frame_equal

source = pd.read_csv(
    "guardian_v2_outputs/freeze_plan_candidate.csv"
)

final = pd.read_csv(
    "guardian_v2_outputs/freeze_plan_explained.csv"
)

# Accept negligible floating-point differences
assert_frame_equal(
    source.reset_index(drop=True),
    final[source.columns].reset_index(drop=True),
    check_exact=False,
    rtol=1e-12,
    atol=1e-9
)

assert len(final) == 10
assert final["acct"].is_unique
assert final["explanation"].notna().all()
assert final["explanation"].str.strip().ne("").all()
assert final["explanation"].str.contains(
    "human approval", case=False
).all()

print("GUARDIAN V2 GRANITE OUTPUT VERIFIED")
print("Accounts:", len(final))
print("Source columns preserved:", len(source.columns))
print("Explanations:", len(final))
print("Human approval included: Yes")
